# 02 — TSI Analysis & Fusion Experiments
## Sensibilidad Temporal de Descriptores de Audio Artesanales

Carga los features pre-extraídos (notebook `01_feature_extraction.ipynb`) y ejecuta el
**método rediseñado (payoff)**:

1. Ganancia de información `G(f,k)` con clasificadores **calibrados** + `L_chance` (base-rate).
2. **TSI por descriptor**: `k*` por **selección anidada**, payoff `G(f,k*) − G(f,k̄)`,
   compuerta `τ` (nula por permutación, re-selecciona argmax), IC bootstrap del payoff,
   `TSI_rel`, sesgo optimista residual.
3. **Matriz de ganancia 7×3** (`G ± σ`, anotada con `k*` y TSI).
4. **5 estrategias de fusión** + referencias (learned-LF sin prior, uniforme).
5. **Validación estadística**: Friedman/Nemenyi (Wilcoxon en IRMAS, K=2),
   Wilcoxon pareado + Bonferroni (C(5,2)=10) + Cliff's δ, Spearman entre rankings.
6. ECE / reliability + importancia PI/MDI (corroborativa).

Escribe `tsi_results.json` y `experiment_results.json` a `RESULTS_ROOT` y los copia a `results/`.

> **Costo:** el barrido anidado entrena muchos clasificadores. Ajusta `N_PERM`, `N_BOOT`,
> `CLASSIFIERS` y el submuestreo de MTAT según el tiempo disponible.

## 0. Setup & Load Cached Features

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies
!pip install -q librosa scikit-learn torch torchaudio tqdm seaborn xgboost scipy scikit-posthocs

# cuML: GPU-accelerated Random Forest (NVIDIA CUDA only). Falls back to sklearn RF.
!pip install -q cuml-cu12 --extra-index-url=https://pypi.nvidia.com 2>/dev/null || echo "cuML not available — sklearn RF will be used"

# Clone the repo to get the src/ modules
import os
REPO_URL = "https://github.com/Gabrieleeh32159/my_paper.git"
REPO_DIR = "/content/my_paper"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

Cloning into '/content/my_paper'...
remote: Enumerating objects: 363, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 363 (delta 54), reused 100 (delta 38), pack-reused 234 (from 1)
Receiving objects: 100% (363/363), 32.01 MiB | 18.36 MiB/s, done.
Resolving deltas: 100% (190/190), done.


In [3]:
import os
import sys
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# === CONFIGURATION ===
REPO_DIR = Path('/content/my_paper')

DRIVE_ROOT = Path('/content/drive/MyDrive/tsi_experiments')
DATA_ROOT = DRIVE_ROOT / 'data'
FEATURES_ROOT = DRIVE_ROOT / 'features'
RESULTS_ROOT = DRIVE_ROOT / 'results'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_DIR / 'experiments'))

DATASET_PATHS = {
    'gtzan': DATA_ROOT / 'gtzan',
    'fma_small': DATA_ROOT,
    'mtat': DATA_ROOT / 'magnatagatune',
    'irmas': DATA_ROOT / 'irmas',
}

SEED = 42
np.random.seed(SEED)
print(f"Features source: {FEATURES_ROOT}")
print(f"Results output:  {RESULTS_ROOT}")

Features source: /content/drive/MyDrive/tsi_experiments/features
Results output:  /content/drive/MyDrive/tsi_experiments/results


In [4]:
%cd {REPO_DIR}/experiments

from src.features import FEATURE_DIMS, SCALES, TRACK_DIM, descriptor_slices
from src.data_loader import get_dataset
from src.classifiers import get_classifier, make_clf_factory
from src.tsi import (
    compute_fold_gains, permutation_null_kstar_gains, gate_threshold,
    tsi_from_fold_gains, apply_gate, select_k_star, SCALE_ORDER, BASELINE,
)
from src.fusion import evaluate_fusion_cv, early_fusion
from src.stats import (
    scale_difference_test, run_pairwise_comparisons, spearman_with_bootstrap_ci,
)
from src.importance import (
    permutation_importance_grouped, early_fusion_groups,
    aggregate_importance_by_descriptor_and_scale,
)
from src.evaluation import expected_calibration_error, reliability_curve, calibration_report

import matplotlib.pyplot as plt
import seaborn as sns

print("All modules loaded.")
print(f"Frame dims: {sum(FEATURE_DIMS.values())} ({FEATURE_DIMS}); track dim/scale: {TRACK_DIM}")

/content/my_paper/experiments
All modules loaded.
Frame dims: 48 ({'mfcc': 20, 'chroma': 12, 'spectral_centroid': 1, 'spectral_contrast': 7, 'spectral_rolloff': 1, 'zcr': 1, 'tonnetz': 6}); track dim/scale: 192


In [5]:
# Load dataset metadata (n_classes, class_names, task_type)
datasets = {}
for name, path in DATASET_PATHS.items():
    try:
        datasets[name] = get_dataset(name, str(path))
        print(f"{name}: {len(datasets[name])} items, {datasets[name].n_classes} classes, "
              f"task={datasets[name].task_type}")
    except Exception as e:
        print(f"{name}: not available ({e})")

gtzan: 1000 items, 10 classes, task=multiclass
fma_small: 8000 items, 8 classes, task=multiclass
mtat: 25863 items, 50 classes, task=multilabel
irmas: 7512 items, 11 classes, task=multiclass


In [6]:
# Load pre-extracted features from Drive (rows aligned by position)
all_features, all_labels, all_splits = {}, {}, {}
for name in list(datasets.keys()):
    short_path = FEATURES_ROOT / f"{name}_short.npy"
    if not short_path.exists():
        print(f"{name}: features not found in {FEATURES_ROOT}; run 01 first. Skipping.")
        continue
    all_features[name] = {
        'short':  np.load(FEATURES_ROOT / f"{name}_short.npy"),
        'medium': np.load(FEATURES_ROOT / f"{name}_medium.npy"),
        'long':   np.load(FEATURES_ROOT / f"{name}_long.npy"),
    }
    raw_labels = np.load(FEATURES_ROOT / f"{name}_labels.npy", allow_pickle=True)
    all_splits[name] = np.load(FEATURES_ROOT / f"{name}_splits.npy", allow_pickle=True)
    if datasets[name].task_type == 'multilabel':
        all_labels[name] = np.stack(raw_labels).astype(np.float32)
    else:
        all_labels[name] = raw_labels.astype(int)
    print(f"{name}: {all_features[name]['short'].shape[0]} tracks loaded "
          f"(labels dtype={all_labels[name].dtype})")
print(f"\nFeatures loaded for: {list(all_features.keys())}")

gtzan: 999 tracks loaded (labels dtype=int64)
fma_small: 7996 tracks loaded (labels dtype=int64)
mtat: 25700 tracks loaded (labels dtype=float32)
irmas: 3756 tracks loaded (labels dtype=int64)

Features loaded for: ['gtzan', 'fma_small', 'mtat', 'irmas']


## 1. Experiment configuration

* **Primary classifier for the TSI** is XGBoost (calibrated). `CLASSIFIERS` can include
  `'rf'`, `'svm'`, `'mlp'` to verify the TSI is not classifier-dependent.
* **Scales**: IRMAS uses **K=2** (`short, medium`) because the 5 s window collapses to the
  3 s clip; all others use the three scales. Baseline `k̄ = medium` (2 s).
* **Folds / protocol (M-5)**: the **TSI uses CV for all datasets** out of statistical
  necessity — the gate `τ` (permutation null), the Friedman test and the payoff bootstrap CI
  all need multiple folds. GTZAN uses 5×10 repeated stratified CV; the official-split datasets
  use a stratified K-fold for that purpose. For the **final fusion point estimate** on
  FMA/MTAT/IRMAS we additionally honour the **official partition** (`USE_OFFICIAL_SPLIT_FOR_FUSION`).
  The powered between-strategy Wilcoxon comes from GTZAN's repeated CV (single official folds
  cannot support a per-fold paired test).
* **Reproducibility (M-1/M-2)**: classifier hyperparameters (incl. the **RF backend actually
  used** — sklearn vs cuML — and XGBoost `max_depth=6, n_estimators=500, lr=0.1`) and all run
  knobs are captured into `run_config.json`.

In [7]:
# --- knobs (scale down for quick runs) ---
PRIMARY_CLF = 'xgb'        # classifier used for the TSI / gain matrix
CLASSIFIERS = ['xgb']      # add 'rf','svm','mlp' to cross-check
N_INNER = 3                # inner folds for nested k* selection
N_BOOT = 1000              # bootstrap resamples for payoff / TSI_rel CIs
N_PERM = 50                # label-permutation replicates for the gate tau (raise to ~200 for the final run if time allows)
MTAT_SUBSAMPLE = None      # e.g. 5000 to cap MTAT track count for speed
USE_OFFICIAL_SPLIT_FOR_FUSION = True  # M-5: also report fusion on the official partition

# --- checkpointing (crash-safe resume for Colab) ---
# The heavy drivers persist progress after every (fold, descriptor)/(perm, fold)
# to CKPT_ROOT on Drive. If the session dies, just re-run all cells: completed
# work is loaded from disk and the sweep continues from where it stopped.
CKPT_ROOT = RESULTS_ROOT / 'checkpoints'
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
RESUME = True              # set False to wipe checkpoints and recompute from scratch
if not RESUME:
    for _f in CKPT_ROOT.glob('*.json'):
        _f.unlink()
    print(f"RESUME=False -> cleared checkpoints in {CKPT_ROOT}")
else:
    print(f"RESUME=True -> using checkpoints in {CKPT_ROOT}")

SCALES_BY_DATASET = {ds: ('short', 'medium') if ds == 'irmas' else tuple(SCALE_ORDER)
                     for ds in all_features}

# `make_clf_factory` is imported from src.classifiers. It wires dataset-specific
# protocol knobs into the classifier — notably the paper's 50% training-track
# subsample for SVM on MTAT (passed as `dataset=name` at the call sites below).

def get_outer_folds(name, y, splits, seed=SEED):
    from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, KFold
    n = len(y)
    if name == 'gtzan':
        rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=seed)
        return list(rskf.split(np.zeros(n), y))
    if datasets[name].task_type == 'multilabel':
        kf = KFold(n_splits=5, shuffle=True, random_state=seed)
        return list(kf.split(np.zeros(n)))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    return list(skf.split(np.zeros(n), y))

def official_split_folds(splits):
    """Single (train+val, test) outer fold from the official split array, or None."""
    splits = np.asarray(splits)
    tr = np.where(np.isin(splits, ['train', 'val']))[0]
    te = np.where(splits == 'test')[0]
    if len(tr) == 0 or len(te) == 0:
        return None
    return [(tr, te)]

print("Config ready. Datasets to process:", list(all_features.keys()))

RESUME=True -> using checkpoints in /content/drive/MyDrive/tsi_experiments/results/checkpoints
Config ready. Datasets to process: ['gtzan', 'fma_small', 'mtat', 'irmas']


## 2. TSI per dataset (nested k*, gate τ, payoff CI, gain matrix)

In [ ]:
def jsonify(stats):
    s = dict(stats)
    s['payoff_ci'] = list(s['payoff_ci'])
    s['tsi_rel_ci'] = list(s['tsi_rel_ci'])
    for k in ('tsi_rel', 'tsi_rel_reported'):
        v = s.get(k)
        if v is None or not np.isfinite(v):
            s[k] = None
    s.pop('k_star_per_fold', None)
    s.pop('payoff_per_fold', None)
    return s

tsi_results = {}
fold_gains_store = {}   # reused by the fusion stage (same nested k*)

for name in all_features:
    feats = all_features[name]
    y = all_labels[name]
    n_classes = datasets[name].n_classes
    task_type = datasets[name].task_type
    scales = SCALES_BY_DATASET[name]
    # dataset-aware factory (wires SVM+MTAT 50% subsample when applicable)
    clf_factory = make_clf_factory(PRIMARY_CLF, dataset=name)

    # optional MTAT subsample for tractability
    idx = np.arange(len(y))
    if name == 'mtat' and MTAT_SUBSAMPLE and len(y) > MTAT_SUBSAMPLE:
        rng = np.random.RandomState(SEED)
        idx = rng.choice(len(y), MTAT_SUBSAMPLE, replace=False)
        feats = {k: feats[k][idx] for k in feats}
        y = y[idx]

    folds = get_outer_folds(name, y, all_splits[name][idx], seed=SEED)
    print(f"\n=== {name}: {len(y)} tracks, {len(folds)} outer folds, scales={scales} ===")

    # 1) per-fold nested gains (progress=True -> nested tqdm bars: folds x descriptors, with % + ETA)
    #    checkpoint_path -> progress saved after every (fold, descriptor); re-run resumes from disk
    fg = compute_fold_gains(feats, y, folds, clf_factory, task_type, n_classes,
                            scales=scales, n_inner=N_INNER, seed=SEED,
                            progress=True, desc=name,
                            checkpoint_path=CKPT_ROOT / f"{name}_{PRIMARY_CLF}_gains.json")
    fold_gains_store[name] = (fg, feats, y, folds, n_classes, task_type, scales,
                              np.asarray(all_splits[name])[idx])

    # 2) per-task gate tau via permutation null (re-selecting argmax each replicate)
    #    progress=True -> tqdm bar over the N_PERM replicates (the slowest phase)
    #    checkpoint_path -> progress saved after every (perm, fold); re-run resumes from disk
    null = permutation_null_kstar_gains(feats, y, folds, clf_factory, task_type, n_classes,
                                        scales=scales, n_permutations=N_PERM, seed=SEED + 7,
                                        progress=True, desc=name,
                                        checkpoint_path=CKPT_ROOT / f"{name}_{PRIMARY_CLF}_null.json")
    tau = gate_threshold(null)

    # 3) TSI stats + gate per descriptor; scale-difference test (Friedman/Wilcoxon)
    descriptors, scale_diff, tsi_by_desc = {}, {}, {}
    for f in FEATURE_DIMS:
        stats = tsi_from_fold_gains(fg[f], baseline=BASELINE, scales=scales,
                                    n_boot=N_BOOT, seed=SEED)
        gated = apply_gate(stats, tau)
        descriptors[f] = jsonify(gated)
        tsi_by_desc[f] = gated['tsi']
        per_fold = {k: [rec['outer'][k] for rec in fg[f]] for k in scales}
        scale_diff[f] = scale_difference_test(per_fold, scales=scales)
        print(f"  {f:18s} TSI={gated['tsi']:.4f} k*={gated['k_star']:6s} "
              f"G(k*)={gated['G_kstar']:.3f} gate={'Y' if gated['gate_pass'] else 'n'} "
              f"exploit={'Y' if gated['exploitable'] else 'n'}")

    tsi_results[name] = {'tau': float(tau), 'baseline': BASELINE,
                         'scales': list(scales), 'n_folds': len(folds),
                         'descriptors': descriptors, 'scale_difference': scale_diff,
                         'tsi_by_descriptor': tsi_by_desc}
print("\nTSI computation complete.")


=== gtzan: 999 tracks, 50 outer folds, scales=('short', 'medium', 'long') ===


gtzan | gains [folds]:   0%|          | 0/50 [00:00<?, ?it/s]

gtzan | gate τ [perm]:   0%|          | 0/50 [00:00<?, ?it/s]

## 3. Five fusion strategies + statistical comparison

In [ ]:
experiment_results = {}

for name in all_features:
    fg, feats, y, folds, n_classes, task_type, scales, splits_aligned = fold_gains_store[name]
    tsi_by_desc = tsi_results[name]['tsi_by_descriptor']
    experiment_results[name] = {}

    for clf_name in CLASSIFIERS:
        factory = make_clf_factory(clf_name, dataset=name)
        print(f"\n=== Fusion: {name} / {clf_name} ===")
        # progress=True -> tqdm bar over outer folds (each fold trains the 5 strategies)
        # checkpoint_path -> per-fold scores saved as each fold finishes; re-run resumes
        fres = evaluate_fusion_cv(feats, y, folds, factory, task_type, n_classes,
                                  fold_gains=fg, tsi_by_descriptor=tsi_by_desc,
                                  scales=scales, n_inner=N_INNER, seed=SEED,
                                  progress=True, desc=f"{name}/{clf_name}",
                                  checkpoint_path=CKPT_ROOT / f"{name}_{clf_name}_fusion.json")

        strategies = {k: {'mean': float(np.mean(v)), 'std': float(np.std(v)),
                          'per_fold': [float(x) for x in v]}
                      for k, v in fres.items()}

        # pairwise Wilcoxon + Bonferroni over the FIVE strategies (learned_lf is a reference)
        five = {k: fres[k] for k in
                ['single_scale', 'early', 'late_uniform', 'tsi_guided', 'tsi_weighted_lf']}
        pairwise = run_pairwise_comparisons(five, alpha=0.05)

        # A-3: upper-bound reference contrast. `learned_lf` (late fusion with
        # weights learned WITHOUT the TSI prior) is the upper bound for the
        # TSI-weighted LF. Reported APART from the Bonferroni family of the 5
        # strategies: a single paired Wilcoxon (uncorrected), as a sanity check
        # of how much the TSI prior costs/gains vs. unconstrained learned weights.
        upper_bound_contrast = run_pairwise_comparisons(
            {'tsi_weighted_lf': fres['tsi_weighted_lf'],
             'learned_lf': fres['learned_lf']}, alpha=0.05)[0]

        # M-3: ECE averaged over ALL outer folds + pooled reliability curve (persisted)
        #      checkpoint_path -> per-fold ECE/predictions saved as each fold finishes
        calib = calibration_report(feats, y, folds, factory, task_type, n_classes, scale='medium',
                                   progress=True, desc=f"{name}/{clf_name}",
                                   checkpoint_path=CKPT_ROOT / f"{name}_{clf_name}_calib.json")

        entry = {'strategies': strategies, 'pairwise': pairwise,
                 'upper_bound_contrast': upper_bound_contrast,
                 'ece': calib['ece_mean'], 'ece_std': calib['ece_std'],
                 'calibration': calib}

        # M-5: also report the 5 strategies on the OFFICIAL partition (point estimate).
        # The TSI itself still uses CV above (needed for the gate/CI). Single official
        # folds cannot support a per-fold Wilcoxon, so only point metrics are stored.
        if USE_OFFICIAL_SPLIT_FOR_FUSION and name != 'gtzan':
            osf = official_split_folds(splits_aligned)
            if osf is not None:
                fg_off = compute_fold_gains(feats, y, osf, factory, task_type, n_classes,
                                            scales=scales, n_inner=N_INNER, seed=SEED,
                                            checkpoint_path=CKPT_ROOT / f"{name}_{clf_name}_official_gains.json")
                fres_off = evaluate_fusion_cv(feats, y, osf, factory, task_type, n_classes,
                                              fold_gains=fg_off, tsi_by_descriptor=tsi_by_desc,
                                              scales=scales, n_inner=N_INNER, seed=SEED,
                                              checkpoint_path=CKPT_ROOT / f"{name}_{clf_name}_official_fusion.json")
                entry['official_split'] = {k: float(v[0]) for k, v in fres_off.items()}
                print("  [official split] " + ", ".join(f"{k}={v[0]:.3f}"
                                                          for k, v in fres_off.items()))

        experiment_results[name][clf_name] = entry
        for k, m in strategies.items():
            print(f"  {k:18s} {m['mean']:.4f} ± {m['std']:.4f}")
        print(f"  ECE={calib['ece_mean']:.4f} ± {calib['ece_std']:.4f}")
        _ub = upper_bound_contrast
        print(f"  [ref upper-bound] tsi_weighted_lf vs learned_lf: "
              f"p={_ub['p_value']:.3f} (uncorrected), "
              f"delta={_ub['cliffs_delta']:+.3f} ({_ub['magnitude']})")
print("\nFusion experiments complete.")

## 4. Consistency of TSI rankings across datasets (Spearman ρ)

In [ ]:
dnames = list(tsi_results.keys())
consistency = {}
for i in range(len(dnames)):
    for j in range(i + 1, len(dnames)):
        a, b = dnames[i], dnames[j]
        va = [tsi_results[a]['tsi_by_descriptor'][f] for f in FEATURE_DIMS]
        vb = [tsi_results[b]['tsi_by_descriptor'][f] for f in FEATURE_DIMS]
        res = spearman_with_bootstrap_ci(va, vb, n_boot=N_BOOT, seed=SEED)
        consistency[f"{a}__vs__{b}"] = {'rho': res['rho'], 'ci': list(res['ci']),
                                        'consistent': res['consistent']}
        print(f"{a} vs {b}: rho={res['rho']:.3f} CI={res['ci']} "
              f"{'consistent' if res['consistent'] else ''}")
for name in tsi_results:
    tsi_results[name]['consistency'] = consistency  # store once (same dict) for reference

## 5. Feature importance (PI / MDI) — corroborative only

In [ ]:
# Permutation importance on the early-fusion (576-d) representation, aggregated to a 7x3
# importance matrix. This COMPLEMENTS (does not govern) the gain matrix / TSI.
RUN_IMPORTANCE = True
if RUN_IMPORTANCE:
    for name in all_features:
        fg, feats, y, folds, n_classes, task_type, scales, splits_aligned = fold_gains_store[name]
        if len(scales) < 3:
            print(f"{name}: skipping 3-scale importance (K={len(scales)})"); continue
        Xe = early_fusion(feats, scales)
        tr, ev = folds[0]
        factory = make_clf_factory(PRIMARY_CLF, dataset=name)
        clf = factory(Xe.shape[1], n_classes, task_type)
        clf.fit(Xe[tr], y[tr])
        groups = early_fusion_groups(scales)
        imp = permutation_importance_grouped(clf, Xe[ev], y[ev], groups,
                                              task_type=task_type, n_repeats=10, seed=SEED)
        matrix = aggregate_importance_by_descriptor_and_scale(imp, scales)
        tsi_results[name]['importance_matrix'] = matrix
        print(f"{name}: importance matrix computed (7x{len(scales)})")
else:
    print("Importance skipped (RUN_IMPORTANCE=False).")

## 6. Save results (RESULTS_ROOT + mirror to results/)

In [ ]:
def _default(o):
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

# M-1 / M-2: reproducibility metadata (classifier backends + hyperparameters + knobs)
def clf_hyperparams(clf_name):
    try:
        c = get_classifier(clf_name, TRACK_DIM, 10, 'multiclass')
        return getattr(c, 'hyperparams', {'note': 'no hyperparams attribute'})
    except Exception as e:
        return {'error': str(e)}

run_config = {
    'seed': SEED, 'primary_clf': PRIMARY_CLF, 'classifiers': CLASSIFIERS,
    'n_inner': N_INNER, 'n_boot': N_BOOT, 'n_perm': N_PERM, 'baseline': BASELINE,
    'mtat_subsample': MTAT_SUBSAMPLE,
    'use_official_split_for_fusion': USE_OFFICIAL_SPLIT_FOR_FUSION,
    'scales_by_dataset': {k: list(v) for k, v in SCALES_BY_DATASET.items()},
    'fold_scheme': {ds: ('5x10 repeated stratified CV' if ds == 'gtzan'
                         else ('5-fold KFold' if datasets[ds].task_type == 'multilabel'
                               else '5-fold stratified CV'))
                    for ds in all_features},
    'classifier_hyperparams': {c: clf_hyperparams(c) for c in CLASSIFIERS},
}
print("run_config:", json.dumps(run_config, indent=2, default=_default))

for target_dir in [RESULTS_ROOT, REPO_DIR / 'experiments' / 'results']:
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with open(target_dir / 'tsi_results.json', 'w') as f:
        json.dump(tsi_results, f, indent=2, default=_default)
    with open(target_dir / 'experiment_results.json', 'w') as f:
        json.dump(experiment_results, f, indent=2, default=_default)
    with open(target_dir / 'run_config.json', 'w') as f:
        json.dump(run_config, f, indent=2, default=_default)
    print(f"Wrote results to {target_dir}")

print("\nNOTE: remember to `git add experiments/results/*.json` to version example outputs.")

## 7. Render tables

In [ ]:
# Pretty Markdown tables (same renderer as results/show.py)
import importlib.util
spec = importlib.util.spec_from_file_location("show", str(REPO_DIR / 'experiments' / 'results' / 'show.py'))
show = importlib.util.module_from_spec(spec); spec.loader.exec_module(show)
print(show.show_tsi(tsi_results))
# Honest-estimator diagnostic: in-sample vs nested TSI payoff + residual optimism
# (the paper asks this to be reported so optimistic k* selection is visible).
print(show.show_residual_optimism(tsi_results))
print(show.show_experiments(experiment_results))

## 8. Gain matrix heatmap (optional visualization)

In [ ]:
# 7x3 gain matrix heatmap per dataset (G mean per cell; k* annotated).
for name in tsi_results:
    scales = tsi_results[name]['scales']
    descs = list(FEATURE_DIMS)
    M = np.array([[tsi_results[name]['descriptors'][f]['gain_matrix'].get(s, {'mean': np.nan})['mean']
                   for s in scales] for f in descs])
    plt.figure(figsize=(4, 5))
    sns.heatmap(M, annot=True, fmt='.2f', xticklabels=scales, yticklabels=descs,
                cmap='viridis', cbar_kws={'label': 'G (truncated)'})
    plt.title(f"Gain matrix — {name}"); plt.tight_layout(); plt.show()